# Content
0.	team members name (and email), 
1.	problem decomposition processes, 
2.	analysis of the data given
3.	planning and designing the algorithm with flow charts or pseudocode, 
4.	implementation by modularizing the components, 
5.	evaluating the algorithm, 
6.	challenges and issues, 
7.  and lastly, conclusion.


# 0 Team Allocation Simulator
Team Members: //rank alphabetically
Name | Email | Role
-- | -- | --
LI Zhuochen | B0255913L@e.ntu.edu.sg | algorithm_designer&programmer
LIU Jinhao | B0255914H@e.ntu.edu.sg | Jupyter_notebook_Transferer
WANG Jiayu | B0255937F@e.ntu.edu.sg | Data_analyzer
WANG Ziyan | B0255943E@e.ntu.edu.sg | flow_chart_drawer
YU Changhan | B0255948A@e.ntu.edu.sg | pseudocode_writer

# 1 problem decomposition processes
- input
    - read file
    - load data
    - initailize
- process
    - main algorithm
- output
    - tag according to original data

# 2 analysis of the data given
*@WANG Jiayu*
![charts](./asset/charts_analyze_data.png)

# 3 planning and designing the algorithm with flow charts or pseudocode
*@YU Chenghan*
```pseudocode
def begin(n):
    if n == 1 :
        for every_student:
            create_team(student);
    endif
    elif n>1
        begin(n-1) # get a previous grouping state
        while number_teams_not_full>1
            rank(estimate_diversity_of_team());
            find_team_to_break():
            if not_full:
                _break_team_into_unallocated_students(team);
    endif
    else:
        raise error
```

*@WANG Ziyan*
![flow_chart](./asset/Flow_chart-WangZiyan_LiZhuochen.png)

# 4 implementation by modularizing the components

In [1]:
class Student:
    """
    Present all the information of a student
    Include str:
    tutorial_group,student_id,name,school,gender,cgpa
    
    一个学生的所有信息
    包括str:
    tutorial_group,student_id,name,school,gender,cgpa
    """
    def __init__(self,tutorial_group="",student_id="",school="",name="",gender="",cgpa:float=0.0):
        """
        初始化函数,
        传入:str / float
        将自己的各种数据按照参数一一赋值
        """
        #深拷贝,养成意识
        from copy import deepcopy
        self.tutorial_group=deepcopy(tutorial_group)
        self.student_id=deepcopy(student_id);
        self.name=deepcopy(name)
        self.school=deepcopy(school)
        self.gender=deepcopy(gender)
        self.cgpa=deepcopy(cgpa);
        # prevent bad data
        try:
            self.cgpa = float(cgpa)
        except:
            self.cgpa = 0.0
        # might not be used, but leave the possibility.
        self.team_assigned=deepcopy(str());
    pass;

    @property
    def info(self):
        return f"{self.tutorial_group},{self.student_id},{self.school},{self.name},{self.gender},{self.cgpa},{self.team_assigned}"


In [2]:
# from student import Student
class Team:# 类 概念
    """
    class stores many Student into students:list
    include function calculate its 3 properties + population
    include function estimate diversity, turn 3 dimensions into 1 number by weighted average

    类:任意个 “Student” 组成一个Team
    students:列表,里面充满学生类
    函数:衡量该分组的性别比例,学校多样性,平均分

    """
    # initialize the overall data, for estimate diversity
    overall_gender_rate:float=1.0;
    overall_school_diversity:float=1.0;
    overall_cgpa_average:float=4.0

    def __init__(self,original_students:list[Student],index):
        # copy data
        from copy import deepcopy
        self.students=list();
        self.students=deepcopy(original_students);

        self.index=index;
        
        #check the content
        for student in self.students:
            if not isinstance(student,Student):
                raise TypeError;
        
        pass;
    
    @property
    def gender_rate(self):
        """
        输出:float性别比例
        count gender in the dictionary
        output Male/Female (both≠0); 0 (either=0)
        """
        count_types={"Male":0,"Female":0};
        for student in self.students:
            if not student.gender in count_types:# if appear wrong data or something else
                raise TypeError;
                # count_types[student.gender]=0;
            count_types[student.gender]+=1;
        
        #evaluate the diversity.
        # for value in list(count_types.values):
        if count_types["Female"]*count_types["Male"]==0:### there is a problem!!
            return 0; #lack one pass all

        numerator=min(count_types["Female"],count_types["Male"]);
        denominator=max(count_types["Female"],count_types["Male"]);

        return numerator/denominator;
        
        pass;
    
    @property
    def school_diversity(self):
        """
        输出:float学校多样性
        学校互不重复的得分最高
        len of set / len of list simply
        """
        count_types={};
        for student in self.students:
            if not student.school in count_types:# if appear wrong data or something else
                count_types[student.school]=0;
            count_types[student.school]+=1;
        
        # evaluate
        try:
            return len(count_types)/len(self.students);
        except ZeroDivisionError:
            return 0;# shall let it report error actually
        pass;
    
    @property
    def cgpa_average(self):
        """
        输出:float平均cgpa
        average
        """
        sum=0;
        for student in self.students:
            sum+=student.cgpa;
        try:
            return sum/len(self.students);
        except ZeroDivisionError:
            return 0;
        pass;
    
    def add_student(self,new_student:Student):
        self.students.append(new_student);
    
    @property
    # def estimate_diversity_of_team(self,overall_gender_rate,overall_school_diversity, overall_cgpa_average):
    def estimate_diversity_of_team(self):
        """
        传出:int该组的得分
        weighted average
        将三个维度的数据化为一个维度,衡量diversity
        p.s.根据要求,性别和学校多样性的权重应该较高.
        """
        WEIGHT_GENDER_RATE=0.7;
        WEIGHT_SCHOOL_DIVERSITY=0.2;
        WEIGHT_CGPA_AVERAGE=0.1;
        
        return self.gender_rate/Team.overall_gender_rate*WEIGHT_GENDER_RATE+self.school_diversity/Team.overall_school_diversity*WEIGHT_SCHOOL_DIVERSITY+(1-abs(self.cgpa_average-Team.overall_cgpa_average))*WEIGHT_CGPA_AVERAGE;
        

        # return self.gender_rate*WEIGHT_GENDER_RATE+self.school_diversity*WEIGHT_SCHOOL_DIVERSITY+self.cgpa_average*WEIGHT_CGPA_AVERAGE;

        pass;
    
    # def __gt__(self,other):
    #     return self.estimate_diversity_of_team>other.estimate_diversity_of_team;

    def __lt__(self,other):
        """
        O:bool diversity comparison
        for sort function convenience
        """
        return self.estimate_diversity_of_team<other.estimate_diversity_of_team;

    pass;

    @property
    def population(self):
        """
        len of students
        """
        return len(self.students);

    def print(self,index="Unknown"):
        print(f"""
### team {index}###
diversity = {self.estimate_diversity_of_team};
gender_rate = {self.gender_rate};school_diversity = {self.school_diversity};cgpa_avg = {self.cgpa_average};
students:{[f"{student.info}" for student in self.students]};
""");

    @property
    def info(self):
        return f"""
@ team {self.index} of {self.population}
diversity = {self.estimate_diversity_of_team};
gender_rate = {self.gender_rate};school_diversity = {self.school_diversity};cgpa_avg = {self.cgpa_average};
students:{[f"{student.info}\n" for student in self.students]};
"""
        pass;

In [3]:
# ⚠️机密 文件:核心算法

#import
# from student import Student
# from team import Team

import logging
logging.basicConfig(
    filename="./asset/team_assigned.csv",
    level=logging.INFO,
    format="%(message)s"
)

class TeamAllocator:
    """
    conduct core algorithms: recursion, greedy
    for N>1 Student:
        revoke self to get the allocation state of N-1
        rank(sort) teams by estimate_diversity_of_team
        break down worst team into unalloc'
        try assign unallo'd student into everyone
        choose the one with highest score after add_student
    for N=1 Student simply everyone's a Team
    """
    
    def __init__(self,original_students:list[Student]):
        """
        initialize
        """
        # copy data
        from copy import deepcopy
        self.students_copy=deepcopy(original_students);

        #set data
        self.teams=list();
        self.unallocated_students=list();
        
        #log the basic diversity or equality of the whole students
        all_in_team=Team(self.students_copy,"all_in_team");
        Team.overall_gender_rate=all_in_team.gender_rate;
        Team.overall_school_diversity=all_in_team.school_diversity;
        Team.overall_cgpa_average=all_in_team.cgpa_average;

        pass;
    
    def _list_teams_not_full(self,team_capacity:int) -> list[Team]:
        """
        I:int
        go thru teams and rec teams needing student
        O: list
        """
        ans=[]
        for team in self.teams:
            if len(team.students)<team_capacity:
                ans.append(team);
        return ans;
    
    def _break_team_into_unallocated_students(self,team_index):
        """
        I:evrything
        get the index and break corresponding team in teams
        pull students into unalloc'
        """
        # print("## _break_team_into_unallocated_students ##")
        for student in self.teams[team_index].students:
            self.unallocated_students.append(student);
        # self.teams[team_index].print();
        del self.teams[team_index];
        pass;
    
    def _assign_unallocated_student_to_team(self,team_capacity):
        """
        for everyone in unalloc':
            try add into every team_not_full, rec highest score
            if there does not exist, create one
            pull into teams
            finally add student

        检查unallocated_students
        如果存在team,将它分到合适的team中
        如果没有,创建一个team
        """
        # print("## _assign_unallocated_student_to_team ##")
        
        # pull-out the first one to assign
        while len(self.unallocated_students):
            #get the first student to allocate
            student=self.unallocated_students.pop(0);

            # rigister the best choice
            best_team=None; 
            best_score=int(-2147483647);

            # get list of teams need student
            teams_need_students= self._list_teams_not_full(team_capacity);
            for new_team in teams_need_students:

                # forsee if the allocation is reasonable
                team_if_add_student=Team(new_team.students+[student],"team_if_add_student")
                new_score=team_if_add_student.estimate_diversity_of_team # estimate the score if the student join
                
                #to compare all the ways of allocating and keep the best one
                if new_score>best_score:
                    best_score=new_score; # record the best choice
                    best_team=new_team;
            
            # print("### this is the best team:")
            # best_team.print();

            if best_team is None:
                best_team=Team([],"best_team");
                self.teams.append(best_team);
                pass;
            
            # finally add the student into the best team
            best_team.add_student(student);

            
            
            # # if no existed team is chosen
            # if len(best_team.students)==1:# this mean that the best choice is the empty one, indicating that teams is empty
            #     self.teams.append(best_team);
            #     pass;

        pass;
    
    def tag_students_with_teams(self,students:list[Student]):
        """
        tag allocation data into required format/file
        go thru required list and add corresponding data
        """
        self.print_result();
        # go trhu original list
        for original_student in students:
            # tag if found
            have_found=False;
            for team_index in range(len(self.teams)):
                for teamed_student in self.teams[team_index].students:
                    # accord with name
                    if original_student.name==teamed_student.name:
                        original_student.team_assigned=str(team_index);
                        # print(original_student.info);
                        have_found=True;
            if have_found!=True:
                print(f"have not found {original_student.info}");
                self.print();
                raise IndexError;
        
        return 1;

    def begin(self,team_capacity:int):
        """
        传入:列表(Student);一个组的人数

        conduct the procedures
        对于要组成X个人的team,将第X个人加入已有X-1个人的组.
        第X个人来自被拆散的组,将最不符合“要求”的组拆散
        """

        # basic statement
        # print(f"# begin{team_capacity}");
        
        if team_capacity==1:
            # erase data
            self.unallocated_students=[];
            self.teams=[];

            # pull all students into unallocated_students
            from copy import deepcopy
            self.unallocated_students=deepcopy(self.students_copy);

            # assign all unall. into teams of itself
            self._assign_unallocated_student_to_team(team_capacity);
            pass;
        
        # advanced statements
        elif team_capacity>1:
            # get the team allocation with 1 less students
            self.begin(team_capacity-1);
            
            # # rank the already-existed teams by overall score
            # self.teams.sort();# leave for further investigation
            # # self.print();
            # self.log(team_capacity);
            
            # # choose the bad team, break and pull into unallo.
            # # while len(self._list_teams_not_full(team_capacity))>1:# if there is unfinished team
            # #     if len(self.teams):
            # #         self._break_team_into_unallocated_students(0);# this is the worst team

            # # assign students in unallo. into teams need a student
            # self._assign_unallocated_student_to_team(team_capacity);

            # complete all not-full team except the last
            while len(self._list_teams_not_full(team_capacity))>1:#avoid indivisible
                # rank by sort(), cmp by estimate_diversity_of_team
                self.teams.sort();
                #
                def find_team_to_break():
                    for i in range(len(self.teams)):
                        if self.teams[i].population<team_capacity:
                            return i;
                    return -1;# to cause error
                    pass;
                #break
                self._break_team_into_unallocated_students(find_team_to_break());
                # assign
                self._assign_unallocated_student_to_team(team_capacity);
                pass;

            pass;
        else:
            # entered <1
            raise ValueError;
    
        # self._tag_students_with_teams();
        
        pass;
    
    # def estimate_diversity_of_team(self,original_team:Team):
    #     """
    #     传入:Team
    #     传出:int该组的得分
    #     将三个维度的数据化为一个维度,衡量diversity
    #     p.s.根据要求,性别和学校多样性的权重应该较高.
    #     """
    #     pass;
    
    def print(self):
        print("unallocated students: ",end="");
        print([student.info for student in self.unallocated_students],end="");
        print("teams:",end="");
        for i in range(len(self.teams)):
            self.teams[i].print(i);
            pass;
    
    def log(self,team_capacity):
        logging.debug(":::::::::::::::::::"+f"TeamAllocator.begin({team_capacity})"+"::::::::::::::::");
        logging.debug(f"""unallocated students: {[student.info for student in self.unallocated_students]}
teams:""")
        for i in range(len(self.teams)):
            logging.debug(self.teams[i].info);
            pass;
        logging.debug("................."+f"end begin"+".................");
        pass;
    
    def print_result(self):
        logging.info(f"index,gender_rate,school_diversity,cgpa_average,*diversity")
        for i in range(len(self.teams)):
            logging.info(f"{i},{self.teams[i].gender_rate},{self.teams[i].school_diversity},{self.teams[i].cgpa_average},{self.teams[i].estimate_diversity_of_team}");
        pass;

    @property
    def info(self):
        for i in range(len(self.teams)):
            self.teams[i].index=i;
        return f"""@TeamAllocator
unallocated students: {self.unallocated_students};
teams: {[self.teams[i].info for i in range(len(self.teams))]}
""";
        pass;

In [4]:
#导入其他文件类
# from team_allocator import TeamAllocator
# from student import Student
# from team import Team

class TeamAllocationSimulator:
    """
    procedures:
        read files, process into data
        process data
        tag data into files, write
    """
    def __init__(self,file_address,output_address):
        """
        TeamAllocationSimulator初始化
        additional: set TEAMCAPACITY as a variable for extendability
        """
        self.file_address=file_address;
        self.output_address=output_address;
        self.TEAM_CAPACITY=5;
        pass;

    # def _parse_lines_from_csv_file(self,original_csv_file:File):
    #     """
    #     传入:源文件,不是地址
    #     传出:列表
    #     源文件每一行作为列表lines的每一项
    #     *无视第一行*
    #     """
    #     from copy import deepcopy
    #     csv_file_copy=deepcopy(original_csv_file);
    #     csv_file_copy.read();

    #     pass;

    def _parse_students_from_lines(self,original_lines) -> list[Student]:
        """
        传入:列表 源文件的各个行
        传出:列表(student类)
        将lines的每一项化成student
        turn each of lines into Student instance
        """
        students=list();
        for line in original_lines:
            line=(line.split('\n'))[0];
            line=line.split(',');
            create_student=Student(line[0],line[1],line[2],line[3],line[4],float(line[5]));
            students.append(create_student);
        return students;
        pass;
    
    # def _parse_teams_into_csv_file(self,original_teams):
    #     """
    #     传入:列表(team类)
    #     传出:
    #     """
    #     pass;
    
    # def _modify_lines_from_teams(self,original_lines,original_teams):
    #     """
    #     传入:列表 源文件的每一列; 列表 team
    #     传出:列表 输出文件的每一列
    #     遍历源文件的每一列,将该列学生所分组按照teams添加新的一列
    #     """
    #     pass;

    def begin(self):
        """
        main body
        apply main procedures
        considering terrible parameter transition, some are not abstracted into func
        """
        # open file:
        lines=[];
        try:
            with open(self.file_address,'r') as csv_file:
                
        # process file to lines:
                lines=csv_file.readlines();
                del lines[0];
                
        except:
            raise FileExistsError;# tbc invalid
            pass;
        
        # process sheet to student list:
        students=self._parse_students_from_lines(lines);

        # into different groups:
        tutorial_groups={};#initialize
        tutorial_group_names=[];#count types
        for s in students:
            if not s.tutorial_group in tutorial_groups:
                tutorial_groups[s.tutorial_group]=[];
                tutorial_group_names.append(s.tutorial_group);#record new tut group
            tutorial_groups[s.tutorial_group].append(s);# add student
        
        #for each tutgroup, call team_allocator
        for tutorial_group_name in tutorial_group_names:
            team_allocator=TeamAllocator(tutorial_groups[tutorial_group_name]);#create instance
            team_allocator.begin(self.TEAM_CAPACITY);
            # tag data into original files form
            is_successful=team_allocator.tag_students_with_teams(tutorial_groups[tutorial_group_name]);
            if not is_successful:
                print("cannot find corresponding student in allocator")
                raise KeyError;
        
        # # call allocator parse team list:
        # team_allocator=TeamAllocator(students);
        # teams=team_allocator.begin(self.TEAM_CAPACITY);

        # # turn list into csv file:
        # new_lines=self._modify_lines_from_teams(lines,teams);

        # create file and put answer in
        try:
            with open(self.output_address,'w') as out_file:
                print("Tutorial Group,Student ID,School,Name,Gender,CGPA,Team Assigned",file=out_file);
                for tutorial_group_name in tutorial_group_names:
                    for student in tutorial_groups[tutorial_group_name]:
                        print(student.info,file=out_file);
            pass;
        except:
            raise FileExistsError;#tbc error
        pass;
    pass;



In [5]:
"""
Team_Allocation_Simulator:main.py
Author:李卓宸 Li "Ary" Zhuochen
所有程序从这里开始,调用写好的TeamAllocationSimulator创建实例,使用方法begin()开始
"""

#常量定义
FILE_ADDRESS="./asset/records.csv";
OUTPUT_ADDRESS="out.csv"

#标准库

#创建实例
# from simulator import TeamAllocationSimulator
team_allocation_simulator=TeamAllocationSimulator(FILE_ADDRESS,OUTPUT_ADDRESS);

#运行
if True:
    team_allocation_simulator.begin();

# 5 evaluating the algorithm

- team_capacity==1:
    - complexity=O($ n $); //assign all students of n into team of 1;
- team_capacity>1:
    - $ nlogn $  for sort();
    - $ \frac{n^2}{4} $ for breaking down and searching for best team

**overall: O(n^2)**


# 6 challenges and issues
- logic in breaking down and assigning:
    - from team of 2 to team of 3, the diversity must descend due to gender
    - so the new team would always be the target of breaking down, which forms a bad cycle of breaking and forming.
    - to avoid this, we need to ignore the team with full capacity.
- the way to calibrate the diversity
    - overall 3 dimension, try to distil into 1;
    - weighted average
    - how to determine if cgpa is good? $ (1-|avg_team-avg_overall)*weight $
    - distribution of weight: 0.7, 0.2, 0.1;

# 7 Conclusion
- This is a successful project due to:
    - concise **decomposition of problem**
    - stable **algorithm** *without randomness*
    - suitable **estimation**
    - high **adaptability** to different situation.
- ~Despite the difficulties in debugging and logging~
- Future work should reduce algorithm complexity, clearer API, more precise description and higher adaptability.
- Thanks to the efforts of th whole team and TA. This project would not be successful without their work and guidance.